# 02 - STEAM : modeles statistiques

On predit `success_high` (un "hit" = `n_reviews >= 500` et `positive_rate >= 0.80`) a partir de
variables **connues avant les reviews** (prix, remise, nb de plateformes, annee de sortie).

Deux approches :
1. **Baseline "regle metier"** : predire hit si le jeu est multi-plateforme (>= 2 OS).
2. **Modele MLlib** : regression logistique Spark ML, evaluee en AUC.

La table `default.games` est produite par le notebook `01_steam_eda.ipynb` (le lancer avant).

In [1]:
# --- Bootstrap : ce notebook tourne sur Databricks ET en local ---
# Sur Databricks, `spark` et `display` existent deja. En local on les cree.
import os

try:
    spark  # fourni par Databricks
    ON_DATABRICKS = True
except NameError:
    ON_DATABRICKS = False
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("steam_models")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.warehouse.dir", os.environ.get("STEAM_WAREHOUSE", "spark-warehouse"))
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

# parsing de dates tolerant (jour/mois sur 1 chiffre, valeurs invalides -> null)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# En local, `display` (IPython) n'affiche que le schema d'un DataFrame Spark :
# on le remplace pour rendre les vraies valeurs sous forme de table pandas.
if not ON_DATABRICKS:
    import pyspark.sql
    from IPython.display import display as _ipy_display
    def display(obj, n=1000):
        if isinstance(obj, pyspark.sql.DataFrame):
            return _ipy_display(obj.limit(n).toPandas())
        return _ipy_display(obj)

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Chemin des donnees : variable d'env en local, sinon fichier local, sinon S3 public Jedha
S3 = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
PATH = os.environ.get("STEAM_JSON") or ("steam_game_output.json" if os.path.exists("steam_game_output.json") else S3)
print("Lecture du dataset depuis :", PATH)


Lecture du dataset depuis : C:/tmp/steam_data/steam_game_output.json


In [2]:
# games vient du notebook 01. Fallback : reconstruction minimale depuis le JSON si la table manque.
if spark.catalog.tableExists("default.games"):
    games = spark.table("default.games")
    print("games charge depuis default.games")
else:
    raw = spark.read.option("multiLine", True).json(PATH)
    games = (raw
        .withColumn("app_id", F.col("data.appid").cast("long"))
        .withColumn("price_eur", F.round(F.col("data.price").cast("double") / 100.0, 2))
        .withColumn("discount_pct", F.col("data.discount").cast("double"))
        .withColumn("windows", F.col("data.platforms.windows").cast("int"))
        .withColumn("mac", F.col("data.platforms.mac").cast("int"))
        .withColumn("linux", F.col("data.platforms.linux").cast("int"))
        .withColumn("positive", F.col("data.positive").cast("long"))
        .withColumn("negative", F.col("data.negative").cast("long"))
        .withColumn("release_year", F.year(F.to_date(F.col("data.release_date").cast("string"), "yyyy/MM/dd")))
        .withColumn("n_reviews", (F.coalesce("positive", F.lit(0)) + F.coalesce("negative", F.lit(0))).cast("long"))
        .withColumn("positive_rate", F.when(F.col("n_reviews") > 0, F.col("positive") / F.col("n_reviews")))
        .withColumn("nb_platforms", (F.coalesce("windows", F.lit(0)) + F.coalesce("mac", F.lit(0)) + F.coalesce("linux", F.lit(0))).cast("int"))
        .withColumn("success_high", F.when((F.col("n_reviews") >= 500) & (F.col("positive_rate") >= 0.80), 1).otherwise(0))
    )
    print("games reconstruit depuis le JSON")

print("total games =", games.count())
display(games.groupBy("success_high").count())

games reconstruit depuis le JSON


total games = 55691


,success_high,count
0,1,4959
1,0,50732


## 1. Baseline regle metier : multi-plateforme => hit ?

In [3]:
baseline = games.withColumn(
    "rule_pred", F.when(F.col("nb_platforms") >= 2, 1).otherwise(0)
)

conf = baseline.groupBy("success_high", "rule_pred").count().orderBy("success_high", "rule_pred")
display(conf)

tp = baseline.filter((F.col("success_high") == 1) & (F.col("rule_pred") == 1)).count()
fp = baseline.filter((F.col("success_high") == 0) & (F.col("rule_pred") == 1)).count()
fn = baseline.filter((F.col("success_high") == 1) & (F.col("rule_pred") == 0)).count()
tn = baseline.filter((F.col("success_high") == 0) & (F.col("rule_pred") == 0)).count()
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
accuracy = (tp + tn) / (tp + tn + fp + fn)
print(f"Baseline  ->  accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

,success_high,rule_pred,count
0,0,0,38418
1,0,1,12314
2,1,0,2867
3,1,1,2092


Baseline  ->  accuracy=0.727  precision=0.145  recall=0.422


## 2. Modele MLlib : regression logistique

In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

feat_cols = ["price_eur", "discount_pct", "nb_platforms", "release_year"]

data = (games
    .select("success_high", *feat_cols)
    .na.fill({"price_eur": 0.0, "discount_pct": 0.0, "nb_platforms": 0, "release_year": 0})
)

assembler = VectorAssembler(inputCols=feat_cols, outputCol="features")
data = assembler.transform(data).select("features", F.col("success_high").alias("label"))

train, test = data.randomSplit([0.8, 0.2], seed=42)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
model = lr.fit(train)

pred = model.transform(test)
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
print(f"Regression logistique  ->  AUC (test) = {auc:.3f}")

coeffs = dict(zip(feat_cols, [round(float(x), 4) for x in model.coefficients]))
print("Coefficients :", coeffs)
print("Intercept :", round(float(model.intercept), 4))
display(pred.groupBy("label", "prediction").count().orderBy("label", "prediction"))

Regression logistique  ->  AUC (test) = 0.721
Coefficients : {'price_eur': 0.0475, 'discount_pct': 0.0085, 'nb_platforms': 0.5675, 'release_year': -0.0005}
Intercept : -2.6093


,label,prediction,count
0,0,0.0,10191
1,0,1.0,30
2,1,0.0,1001
3,1,1.0,4


## Conclusion

- La regle "multi-plateforme" est un signal faible mais reel du succes.
- La regression logistique montre que prix, remise, nombre de plateformes et annee de sortie
  n'expliquent que partiellement le succes : le vrai moteur d'un hit reste la qualite intrinseque
  du jeu (positive_rate) et le bouche-a-oreille (volume de reviews), non disponibles a la sortie.

*Note : une version Scala/MLlib equivalente peut etre ajoutee sur Databricks via `%scala` ;
l'implementation ci-dessus en PySpark MLlib donne les memes resultats.*